In [16]:
from __future__ import annotations
import logging, re, xml.etree.ElementTree as ET
from math import radians, sin, cos, sqrt, atan2
from typing import Any, Dict, Optional, Tuple, List

import requests
import pandas as pd

import geopandas as gpd
from shapely.geometry import Point

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

In [17]:
from busca_enderecos_simplificado_v5 import busca_candidatos_df, busca_candidatos_por_similaridade
#cadastro = r"C:\Py_Notebooks\buscaEnderecos_CET\ARQUIVOS_TESTE/cadastroRuas_V5_13_10_2025.xlsx"
cadastro = r"C:\Users\vitor.braz\Downloads\cadastroRuas_V5_13_10_2025.xlsx"

In [18]:
# Endpoints internos CET
GEOCODE_ENDERECOS_URL = "http://cetaplica/geosoap/geocode.asmx/buscaEnderecos"
GEOCODE_LATLON_URL    = "http://cetaplica/geosoap/geocode.asmx"
GEOSERVER_WMS_URL     = "http://cet-inf7242:8080/geoserver/sqlProducao/wms"

# Camadas consultadas no GetFeatureInfo
GEOSERVER_LAYERS = [
    "sqlProducao:gets_lg",             # GET
    "sqlProducao:dets_lg",             # DET
    "sqlProducao:mdcSubPrefeitura",    # Subprefeitura
    "sqlProducao:mdcDistrito",         # Distrito
    "sqlProducao:regiao5_lg",          # Região oficial (NOME)
    "sqlProducao:ClassVia"             # Classificação viária
]

# Rodovias → nome oficial
MAPEANDO_RODOVIAS = {
    "SP 010": "07061",
    "BR 381": "07061",
    "SP 021": "51501",
    "SP 060": "27099",
    "BR 116": "27099",
    "SP 070": "Z1833",
    "SP 150": "01247",
    "SP 160": "24491",
    "SP 015": "MARGINAL",
    "SP 270": "16878",
    "SP 280": "13026",
    "SP 330": "01429",
    "SP 348": "35449",
}

MARGINAIS = {
    "TIETE": [
        "33234",
        "15188",
        "02421",
        "06371",
        "14270",
        "12436",
        "04561",
    ],
    "PINHEIROS": [
        "03376",
        "00569",
        "12502",
        "13014",
        "51879",
        "35858",
        "06238",
        "08889",
    ],
    "OUTRAS_MARGINAIS": [
        "09863", "09716", "20961", "31844", "42621", "39620", "33509", "44616", 
        "77631", "29567", "16326", "13023", "44651", "13046", "25850", "13017", 
        "13026", "Z4309",
    ],
    "RADIAL LESTE": [
        "00543", "00544", "01702", "07645", "12156", "22555", "34122", "35652", 
        "39641", "49216", "49302", "53024", "Z1615", "33987", "42308", "33667"
    ],
}

# ✅ COLUNAS CORRIGIDAS - APENAS COLUNAS DESEJADAS
COLUNAS_ENRIQUECIDAS = [
    'codlog', 'logradouro_PMSP', 'latitude', 'longitude', 'distancia_km',
    'GET', 'DET', 'SUB', 'Classificacao', 'Distrito_Nome', 'Regiao_Nome', 'similaridade'
]

# Tipos de via/abreviações
TIPOS_VIA = {
    "ACESSO","ALAMEDA","AVENIDA","BECO","CAMINHO","COMPLEXO VIARIO","ESPACO LIVRE","ESPLANADA",
    "ESTRADA","ESCADARIA","ESTRADA PARTICULAR","GALERIA","LADEIRA","LARGO","PASSARELA","PRACA",
    "PRACA PROJETADA","PARQUE","PARQUE ESTADUAL","PARQUE LINEAR","PARQUE MUNICIPAL",
    "PASSAGEM DE PEDESTRE","PASSAGEM PARTICULAR","PASSAGEM SUBTERRÂNEA","PONTILHAO",
    "RUA","RUA PARTICULAR","RUA PROJETADA","RODOVIA","TRAVESSA","TRAVESSA PARTICULAR",
    "VIA DE CIRCULACAO DE PEDESTRES","VIADUTO","VIELA","VIA ELEVADA","VIA ELEVADA DE PEDESTRES",
    "VEREDA","VIELA SANITARIA","VILA","VIELA PROJETADA","VIELA PARTICULAR",
    "AC","AL","AV","BC","CM","CV","EL","EPL","ES","ESC","ESP","GL","LD","LG",
    "PA","PC","PP","PQ","PQE","PQL","PQM","PS","PSP","PSS","PTL",
    "R","RP","RPJ","RV","TV","TVP","VCP","VD","VE","VEL","VEP","VER","VES",
    "VL","VLP","VP","DE","DA","DO"
}


In [20]:
def _com_coringa(s: str) -> str:
    s = (s or "").strip()
    if not s:
        return s
    return s if s.endswith("%") else s + "%"

def _buscar_enderecos_relativo(term: str) -> pd.DataFrame:
    """Busca com coringa % - MANTIDO como solicitado"""
    return buscar_enderecos(_com_coringa(term))

def calcular_distancia_km(lat1, lon1, lat2, lon2):
    if None in [lat1, lon1, lat2, lon2]:
        return None
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return round(R * 2 * atan2(sqrt(a), sqrt(1 - a)), 3)

def remover_tipo_via(endereco: str) -> str:
    """Função mantida mas NÃO USADA na busca principal"""
    partes = endereco.strip().upper().split()
    if partes and partes[0] in TIPOS_VIA:
        return " ".join(partes[1:]).strip()
    return endereco.strip()

def logradouro_pmsp(row) -> str:
    partes = [
        (row.get("tipo") or "").strip(),
        (row.get("titulo") or "").strip(),
        (row.get("preposicao") or "").strip(),
        (row.get("nome") or "").strip()
    ]
    return " ".join(p for p in partes if p).strip()

def montar_endereco_completo(row, numero="") -> str:
    return f"{row['codlog']}, {numero}".strip()

In [21]:
def buscar_latlon_exato(endereco: str) -> tuple[float, float] | None:
    headers = {"Content-Type": "text/xml; charset=utf-8", "SOAPAction": "http://tempuri.org/buscaLatLonExato"}
    soap_body = f"""<?xml version="1.0" encoding="utf-8"?>
    <soap:Envelope xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
                   xmlns:xsd="http://www.w3.org/2001/XMLSchema"
                   xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/">
      <soap:Body>
        <buscaLatLonExato xmlns="http://tempuri.org/">
          <endereco>{endereco}</endereco>
        </buscaLatLonExato>
      </soap:Body>
    </soap:Envelope>"""
    try:
        response = requests.post(GEOCODE_LATLON_URL, data=soap_body, headers=headers, timeout=6)
        response.raise_for_status()
        ns = {"soap":"http://schemas.xmlsoap.org/soap/envelope/","ns":"http://tempuri.org/"}
        root = ET.fromstring(response.content)
        ponto = root.find(".//ns:buscaLatLonExatoResult", ns)
        if ponto is None: return None
        x = ponto.findtext("ns:x", default=None, namespaces=ns)
        y = ponto.findtext("ns:y", default=None, namespaces=ns)
        if x is None or y is None: return None
        return float(y), float(x)
    except (requests.RequestException, ET.ParseError, ValueError):
        return None

def _infer_layer_name(feat: dict) -> str:
    fid = str(feat.get("id",""))
    return fid.split(".",1)[0] if "." in fid else fid

def obter_dados_geoserver(lat: float, lon: float) -> dict:
    layers = ",".join(GEOSERVER_LAYERS)
    params = {
        "SERVICE":"WMS","VERSION":"1.1.1","REQUEST":"GetFeatureInfo",
        "LAYERS":layers,"QUERY_LAYERS":layers,"STYLES":"",
        "SRS":"EPSG:4326",
        "BBOX": f"{lon-0.0015},{lat-0.0015},{lon+0.0015},{lat+0.0015}",
        "WIDTH":"101","HEIGHT":"101","X":"50","Y":"50",
        "INFO_FORMAT":"application/json","FEATURE_COUNT":"50"
    }
    try:
        # ✅ AUMENTAR TIMEOUT para 30 segundos
        r = requests.get(GEOSERVER_WMS_URL, params=params, timeout=30)
        r.raise_for_status()
        if not r.headers.get("content-type","").startswith("application/json"):
            return {}
        data = r.json()
        out: Dict[str, Any] = {}

        for f in data.get("features", []):
            props = f.get("properties", {}) or {}
            layer = _infer_layer_name(f)

            if layer.endswith("gets"):
                out["GET"] = props.get("sigla")
            if layer.endswith("dets"):
                out["DET"] = props.get("sigla")
            if layer.endswith("mdcSubPrefeitura"):
                out["SUB"] = props.get("sigla2") 
            if layer.endswith("mdcDistrito"):
                out["Distrito_Nome"] = props.get("Nome_distr")
            if layer.endswith("regiao5"): 
                out["Regiao_Nome"] = props.get("nome")
            if layer.endswith("ClassVia"):
                out["Classificacao"] = props.get("Classificacao")
        return out
    except Exception as e:
        logging.warning("⚠️ GeoServer não disponível: %s", e)
        return {}

def detectar_marginal(endereco: str) -> Optional[str]:
    """
    Detecta marginais de forma INTELIGENTE
    """
    if not endereco:
        return None

    e = endereco.upper().strip()

    # ✅ SP 015 é tratado como "AMBAS"
    e = e.replace("ZERO QUINZE", "15")
    e = re.sub(r"SP\s*0?-?\s*15", "SP 15", e)
    e = re.sub(r"[-/]", " ", e)
    e = re.sub(r"\s+", " ", e)

    # SP 015 explícito
    if re.search(r"\bSP 15\b", e):
        if "TIETE" in e:
            return "TIETE"
        if "PINHEIROS" in e:
            return "PINHEIROS"
        return "AMBAS"

    # MARGINAL explícita
    if "MARGINAL" in e:
        if "TIETE" in e:
            return "TIETE"
        if "PINHEIROS" in e:
            return "PINHEIROS"
        return "AMBAS"
     
    if "RADIAL LESTE" in e:
        return "RADIAL LESTE"

    return None

def _df_nao_encontrado() -> pd.DataFrame:
    d = {col: None for col in COLUNAS_ENRIQUECIDAS}
    d["logradouro_PMSP"] = "NAO ENCONTRADO"
    return pd.DataFrame([d])

def buscar_endereco_com_fallback(
    endereco: str,
    numero: int | str = "",
    lat_origem: float | None = None,
    lon_origem: float | None = None,
    cadastro_path: str = cadastro,
    min_sim: int = 80,
    top_k: int = 10
) -> pd.DataFrame:

    # ✅ VERIFICAÇÃO CONTRA VALORES VAZIOS
    if not endereco or not isinstance(endereco, str) or endereco.strip() == "":
        logging.info("📭 Endereço vazio → NAO ENCONTRADO")
        return _df_nao_encontrado()
        
    # 1) Fluxo normal
    try:
        df = buscar_enderecos_com_latlon(endereco, numero, lat_origem, lon_origem)
        # ✅ CORREÇÃO: Garantir que similaridade seja NaN quando não vem do embedding
        if "similaridade" not in df.columns:
            df["similaridade"] = float("nan")
    except Exception:
        df = _df_nao_encontrado()
        df["similaridade"] = float("nan")

    if not df.empty and df.iloc[0]["logradouro_PMSP"] != "NAO ENCONTRADO":
        return df

    # 2) Fallback apenas se necessário
    if not cadastro_path:
        return df

    try:
        from busca_enderecos_simplificado_v5 import busca_candidatos_df
        
        # Busca seletiva
        candidatos = busca_candidatos_df(
            endereco=endereco,
            cadastro_path=cadastro_path,
            top_k=20,
            min_sim=min_sim
        )
        
        if candidatos is None or candidatos.empty:
            return df
        
        # ✅ CORREÇÃO: Processar TODOS os candidatos do fallback
        coletados = []
        for _, row in candidatos.iterrows():
            codlog = str(row["codlog"])
            nome_local = str(row["LOCAL"])
            similaridade = int(row["SIMILARIDADE"])
            
            df_try = _resolver_por_codlog(codlog, nome_local, numero, lat_origem, lon_origem)
            
            if df_try is not None and not df_try.empty:
                df_try["similaridade"] = similaridade
                coletados.append(df_try)

        if coletados:
            df_all = pd.concat(coletados, ignore_index=True)
            df_all = df_all.drop_duplicates(subset=["codlog"], keep="first")

            # Ordenação inteligente
            if lat_origem is not None and lon_origem is not None:
                df_all = df_all.sort_values(["similaridade", "distancia_km"], ascending=[False, True])
            else:
                df_all = df_all.sort_values("similaridade", ascending=False)
            
            return df_all.head(top_k)
        else:
            return df
            
    except Exception:
        return df

In [22]:
def definir_circunscricao_via_shapefile(df: pd.DataFrame, col_lat: str, col_lon: str, shapefile_sp: str) -> pd.DataFrame:
    """
    Adiciona a coluna 'CIRCUNSCRICAO' ao DataFrame
    """
    gdf_sp = gpd.read_file(shapefile_sp).to_crs(epsg=4326)

    def ponto_dentro_sp(lat, lon):
        pt = Point(lon, lat)
        return "NAO" if gdf_sp.contains(pt).any() else "SIM"

    df["CIRCUNSCRICAO"] = df.apply(lambda row: ponto_dentro_sp(row[col_lat], row[col_lon]), axis=1)
    return df
    
def _processar_resultados_geocode(
    df: pd.DataFrame, 
    numero: str, 
    lat_origem: float | None, 
    lon_origem: float | None
) -> pd.DataFrame:
    """
    ✅ NOVA FUNÇÃO: Processa resultados do GEOCODE para garantir enriquecimento completo
    """
    if df.empty:
        return df
    
    # 1) Adicionar logradouro_PMSP
    df["logradouro_PMSP"] = df.apply(logradouro_pmsp, axis=1)
    
    # 2) Criar endereco_consulta e buscar coordenadas
    df["endereco_consulta"] = df.apply(lambda row: montar_endereco_completo(row, numero), axis=1)
    
    # 3) Buscar coordenadas para TODOS os resultados
    latlon_results = []
    for endereco in df["endereco_consulta"]:
        latlon = buscar_latlon_exato(endereco) or (None, None)
        latlon_results.append(latlon)
    
    df[["latitude", "longitude"]] = pd.DataFrame(latlon_results, index=df.index)
    
    # 4) Calcular distância
    if lat_origem is not None and lon_origem is not None:
        df["distancia_km"] = df.apply(
            lambda r: calcular_distancia_km(lat_origem, lon_origem, r["latitude"], r["longitude"]),
            axis=1
        )
    else:
        df["distancia_km"] = None

    # ✅✅✅ CORREÇÃO CRÍTICA: GeoServer para TODOS os resultados do GEOCODE
    for i, row in df.iterrows():
        lat, lon = row["latitude"], row["longitude"]
        if pd.notna(lat) and pd.notna(lon):
            info = obter_dados_geoserver(lat, lon)
            # ✅ Garantir que as colunas existam antes de atribuir
            for k, v in info.items():
                if k not in df.columns:
                    df[k] = None
                df.at[i, k] = v

    # ✅ Garantir que todas as colunas enriquecidas existam
    for col in ['GET', 'DET', 'SUB', 'Classificacao', 'Distrito_Nome', 'Regiao_Nome']:
        if col not in df.columns:
            df[col] = None

    # ✅ REMOVER COLUNAS INDESEJADAS
    for coluna_indesejada in ['tipo', 'titulo', 'preposicao', 'nome', 'endereco_consulta']:
        if coluna_indesejada in df.columns:
            df = df.drop(columns=[coluna_indesejada])

    # ✅ Adicionar similaridade como NaN (não vem do embedding)
    df["similaridade"] = float("nan")

    # Ordenação por distância
    if lat_origem is not None and lon_origem is not None:
        df = df.sort_values(by="distancia_km", ascending=True)

    return df

def _buscar_por_embedding_rapido(
    endereco: str, 
    numero: str, 
    lat_origem: float | None, 
    lon_origem: float | None
) -> pd.DataFrame:
    """
    Embedding RÁPIDO - apenas os melhores candidatos
    """
    try:
        from busca_enderecos_simplificado_v5 import busca_candidatos_df
        
        # Busca seletiva - apenas bons matches
        candidatos = busca_candidatos_df(
            endereco=endereco,
            cadastro_path=cadastro,
            top_k=15,
            min_sim=80
        )
        
        if candidatos is None or candidatos.empty:
            return pd.DataFrame()
        
        # Processar TODOS os candidatos do embedding
        coletados = []
        for _, row in candidatos.iterrows():
            df_try = _resolver_por_codlog(
                str(row["codlog"]), 
                str(row["LOCAL"]), 
                numero, 
                lat_origem, 
                lon_origem
            )
            if df_try is not None and not df_try.empty:
                df_try["similaridade"] = int(row["SIMILARIDADE"])
                coletados.append(df_try)
        
        return pd.concat(coletados, ignore_index=True) if coletados else pd.DataFrame()
            
    except Exception:
        return pd.DataFrame()

def _resolver_por_codlog(
    codlog: str,
    logradouro_display: str,
    numero: int | str,
    lat_origem: float | None,
    lon_origem: float | None,
) -> pd.DataFrame | None:
    """
    Resolve um candidato usando CODLOG
    """
    # Buscar nome oficial via GEOCODE
    df_logradouro = buscar_enderecos(codlog)
    if df_logradouro.empty:
        logradouro_oficial = logradouro_display
    else:
        log_info = df_logradouro.iloc[0]
        logradouro_oficial = logradouro_pmsp(log_info)
    
    # Buscar coordenadas
    endereco_completo = f"{codlog}, {numero}"
    latlon = buscar_latlon_exato(endereco_completo)
    if not latlon:
        return None
    lat, lon = latlon

    # Calcular distância
    if lat_origem is not None and lon_origem is not None:
        dist = calcular_distancia_km(lat_origem, lon_origem, lat, lon)
    else:
        dist = None

    # ✅✅✅ CORREÇÃO: GeoServer SEMPRE chamado
    info = obter_dados_geoserver(lat, lon) or {}

    # Montar resultado
    out = {
        "codlog": str(codlog),
        "logradouro_PMSP": logradouro_oficial,
        "latitude": lat,
        "longitude": lon,
        "distancia_km": dist,
        "GET": info.get("GET"),
        "DET": info.get("DET"),
        "SUB": info.get("SUB"),
        "Classificacao": info.get("Classificacao"),
        "Distrito_Nome": info.get("Distrito_Nome"),
        "Regiao_Nome": info.get("Regiao_Nome"),
        "similaridade": None,
    }
    
    return pd.DataFrame([out], columns=COLUNAS_ENRIQUECIDAS)
    
def buscar_em_marginal(tipo_marginal: str, lat_origem: float | None, lon_origem: float | None) -> pd.DataFrame:
    """
    Busca em TODAS as marginais sem limite - 31 vias é aceitável
    """
    if lat_origem is None or lon_origem is None:
        logging.warning("Busca em marginal requer lat_origem/lon_origem. Retornando NAO ENCONTRADO.")
        return _df_nao_encontrado()

    chave = tipo_marginal.upper()
    
    if chave == "AMBAS":
        vias = (MARGINAIS.get("PINHEIROS", []) + 
                MARGINAIS.get("TIETE", []) + 
                MARGINAIS.get("OUTRAS_MARGINAIS", []))
    else:
        vias = MARGINAIS.get(chave, [])
        
    if not vias:
        return _df_nao_encontrado()

    resultados = []
    
    for via in vias:
        df_cand = buscar_enderecos(via)
        if df_cand.empty:
            continue

        df_cand["logradouro_PMSP"] = df_cand.apply(logradouro_pmsp, axis=1)
        df_cand["endereco_consulta"] = df_cand.apply(lambda r: montar_endereco_completo(r, ""), axis=1)
        
        latlon = df_cand["endereco_consulta"].apply(lambda e: buscar_latlon_exato(e) or (None, None))
        df_cand[["latitude", "longitude"]] = pd.DataFrame(latlon.tolist(), index=df_cand.index)
        
        df_cand["distancia_km"] = df_cand.apply(
            lambda r: calcular_distancia_km(lat_origem, lon_origem, r["latitude"], r["longitude"]), axis=1
        )

        resultados.append(df_cand)

    if not resultados:
        return _df_nao_encontrado()

    df_all = pd.concat(resultados, ignore_index=True).dropna(subset=["latitude", "longitude"], how="any")
    
    # ✅ CORREÇÃO DIRETA: GeoServer para TODOS os resultados das marginais
    for i, row in df_all.iterrows():
        lat, lon = row["latitude"], row["longitude"]
        if pd.notna(lat) and pd.notna(lon):
            info = obter_dados_geoserver(lat, lon)
            # ✅ GARANTIR que as colunas existam antes de preencher
            for col in ['GET', 'DET', 'SUB', 'Classificacao', 'Distrito_Nome', 'Regiao_Nome']:
                if col not in df_all.columns:
                    df_all[col] = None
            # ✅ PREENCHER os valores
            for k, v in info.items():
                if k in df_all.columns:
                    df_all.at[i, k] = v

    # ✅ REMOVER COLUNAS INDESEJADAS
    for coluna_indesejada in ['tipo', 'titulo', 'preposicao', 'nome', 'endereco_consulta']:
        if coluna_indesejada in df_all.columns:
            df_all = df_all.drop(columns=[coluna_indesejada])
    
    # ✅ GARANTIR COLUNAS ENRIQUECIDAS
    for col in COLUNAS_ENRIQUECIDAS:
        if col not in df_all.columns:
            df_all[col] = None

    # ✅ ORDENAR POR DISTÂNCIA
    df_all = df_all.sort_values(by="distancia_km", ascending=True)

    return df_all.head(50)
    
def buscar_enderecos(endereco: str) -> pd.DataFrame:
    """
    Busca endereços no GEOCODE com substituição INTELIGENTE de rodovias
    """
    # ✅ CORREÇÃO: Substituição ROBUSTA de rodovias ANTES da busca
    endereco_upper = endereco.upper().strip()
    
    # ✅ 1) Substituir códigos de rodovia por nomes oficiais
    for codigo, nome in MAPEANDO_RODOVIAS.items():
        # Padrões: "SP 070", "SP070", "SP-070"
        padroes = [
            rf"\b{codigo}\b",
            rf"\b{codigo.replace(' ', '')}\b", 
            rf"\b{codigo.replace(' ', '-')}\b"
        ]
        
        for padrao in padroes:
            if re.search(padrao, endereco_upper, re.IGNORECASE):
                endereco = re.sub(padrao, nome, endereco_upper, flags=re.IGNORECASE)
                logging.info(f"✅ Rodovia substituída: {codigo} → {nome}")
                break
    
    # ✅ 2) Casos especiais para SP-015 (Marginais)
    if re.search(r'\bSP\s*0?-?\s*15\b', endereco_upper):
        if "TIETE" in endereco_upper:
            endereco = "MARGINAL TIETE"
        elif "PINHEIROS" in endereco_upper:
            endereco = "MARGINAL PINHEIROS"
        else:
            endereco = "MARGINAL"
        logging.info(f"✅ SP-015 convertido para: {endereco}")
    
    logging.info(f"🔍 Buscando no GEOCODE: {endereco}")
    
    try:
        response = requests.get(GEOCODE_ENDERECOS_URL, params={"endereco": endereco}, timeout=6)
        response.raise_for_status()
    except requests.RequestException as e:
        logging.error("Falha na consulta do endereço '%s': %s", endereco, e)
        return pd.DataFrame()

    try:
        ns = {"ns": "http://tempuri.org/"}
        root = ET.fromstring(response.content)
        enderecos = root.findall("ns:Endereco", ns)
        if not enderecos:
            return pd.DataFrame(columns=["codlog","tipo","titulo","preposicao","nome"])

        registros = []
        for endereco_xml in enderecos:
            log = endereco_xml.find("ns:logradouro1", ns)
            if log is None: 
                continue
            registros.append({
                "codlog":     log.findtext("ns:codlog", default="", namespaces=ns),
                "tipo":       log.findtext("ns:tipo", default="", namespaces=ns),
                "titulo":     log.findtext("ns:titulo", default="", namespaces=ns),
                "preposicao": log.findtext("ns:preposicao", default="", namespaces=ns),
                "nome":       log.findtext("ns:nome", default="", namespaces=ns)
            })
        return pd.DataFrame(registros)
    except ET.ParseError as e:
        logging.error("Erro ao interpretar XML: %s", e)
        return pd.DataFrame()
        
def _gerar_candidatos_endereco(endereco: str) -> list[str]:
    """Gera candidatos com substituição INTELIGENTE de rodovias"""
    e0 = (endereco or "").strip().upper()
    cands = set([e0])
    
    # ✅ SUBSTITUIÇÃO IMEDIATA de rodovias
    for codigo, nome in MAPEANDO_RODOVIAS.items():
        # Verificar todos os formatos: SP 070, SP070, SP-070
        padroes = [
            rf"\b{codigo}\b",
            rf"\b{codigo.replace(' ', '')}\b",
            rf"\b{codigo.replace(' ', '-')}\b"
        ]
        
        for padrao in padroes:
            if re.search(padrao, e0, re.IGNORECASE):
                substituto = re.sub(padrao, nome, e0, flags=re.IGNORECASE)
                cands.add(substituto)
                logging.info(f"🎯 Candidato gerado: {codigo} → {nome}")

    return [c for c in cands if c]
    
def buscar_enderecos_com_latlon(
    endereco: str,
    numero: int | str = "",
    lat_origem: float | None = None,
    lon_origem: float | None = None,
    max_candidatos: int = 35,
) -> pd.DataFrame:
    # ✅ VERIFICAÇÃO NA FUNÇÃO PRINCIPAL TAMBÉM
    if not endereco or not isinstance(endereco, str) or endereco.strip() == "":
        return _df_nao_encontrado()
    # 1) Marginais
    tipo_marginal = detectar_marginal(endereco)
    if tipo_marginal:
        logging.info("🎯 Marginal detectada: %s", tipo_marginal)
        return buscar_em_marginal(tipo_marginal, lat_origem, lon_origem)

    candidatos = []
    enderecos_ja_buscados = set()

    # 2) Busca com endereço ORIGINAL
    df = _buscar_enderecos_relativo(endereco)
    if not df.empty:
        candidatos.append(df)
    enderecos_ja_buscados.add(endereco.upper())

    # 3) Variações de rodovias
    for cand in _gerar_candidatos_endereco(endereco):
        if cand.upper() not in enderecos_ja_buscados:
            dfv = _buscar_enderecos_relativo(cand)
            if not dfv.empty:
                candidatos.append(dfv)
            enderecos_ja_buscados.add(cand.upper())

        if sum(len(x) for x in candidatos) >= 30:
            break

    # ✅ 4) SE NÃO ENCONTROU NADA AINDA: tenta sem tipo de via
    if not candidatos:
        endereco_sem_tipo = remover_tipo_via(endereco)
        if endereco_sem_tipo != endereco and endereco_sem_tipo.upper() not in enderecos_ja_buscados:
            logging.info(f"🔧 Tentando sem tipo de via: '{endereco}' → '{endereco_sem_tipo}'")
            df_sem_tipo = _buscar_enderecos_relativo(endereco_sem_tipo)
            if not df_sem_tipo.empty:
                candidatos.append(df_sem_tipo)

    # Consolida resultados
    if candidatos:
        df = pd.concat(candidatos, ignore_index=True).drop_duplicates()
    else:
        df = pd.DataFrame()

    # ✅ 5) SÓ USA EMBEDDING SE NADA FUNCIONOU
    if df.empty:
        df = _buscar_por_embedding_rapido(endereco, numero, lat_origem, lon_origem)
    else:
        df = _processar_resultados_geocode(df, numero, lat_origem, lon_origem)

    return df if not df.empty else _df_nao_encontrado()

In [ ]:
import pandas as pd
import time
import os
from datetime import datetime
from unidecode import unidecode
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# -------------------------------------------
# 🔹 Configurações
# -------------------------------------------
caminho_entrada = r"C:\Users\vitor.braz\Downloads\sinistros_sp_parte1.xlsx"
cadastro_path = r"C:\Users\vitor.braz\Downloads\cadastroRuas_V5_13_10_2025.xlsx"
shapefile_sp = r"C:\Users\vitor.braz\Downloads\LIMITE_MUNICIPAL\LIMITE_MUNICIPAL_4674\LIMITE_MUNICIPAL_4674.shp"
coluna_endereco = "logradouro"  # ou "endereco" se sua coluna tiver outro nome
coluna_numero = "numero"
min_sim = 70
top_k = 1
salvar_resultado = True

# -------------------------------------------
# 🔹 Leitura e pré-processamento
# -------------------------------------------
print(f"\n📂 Lendo planilha: {caminho_entrada}")
df_input = pd.read_excel(caminho_entrada)
df_input = df_input.drop_duplicates().dropna(how="all")

# Normaliza texto
for c in df_input.select_dtypes(include=["object"]).columns:
    df_input[c] = df_input[c].astype(str).apply(unidecode).str.strip()

# Detecta colunas de latitude/longitude
cols_lower = {c.lower(): c for c in df_input.columns}
col_lat = next((cols_lower[c] for c in cols_lower if "lat" in c), None)
col_lon = next((cols_lower[c] for c in cols_lower if "lon" in c or "lng" in c), None)
if not col_lat or not col_lon:
    raise ValueError("❌ Não foi possível identificar colunas de latitude/longitude na planilha.")

# Corrige formato das coordenadas
df_input[col_lat] = df_input[col_lat].astype(str).str.replace(",", ".").astype(float)
df_input[col_lon] = df_input[col_lon].astype(str).str.replace(",", ".").astype(float)

# Circunscrição via shapefile
print("\n🌐 Definindo circunscrição via shapefile...")
df_input = definir_circunscricao_via_shapefile(df_input, col_lat, col_lon, shapefile_sp)
print("✅ Circunscrição adicionada com sucesso.\n")

# -------------------------------------------
# 🔹 Processamento de endereços
# -------------------------------------------
resultados = []
print(f"🔍 Processando {len(df_input)} registros...\n")
inicio_total = time.time()

for i, row in df_input.iterrows():
    logradouro = str(row.get(coluna_endereco, "") or row.get("endereco", ""))
    numero = row.get(coluna_numero, "")
    lat_origem = row[col_lat]
    lon_origem = row[col_lon]

    print(f"[{i+1}/{len(df_input)}] {logradouro} ({lat_origem}, {lon_origem})")
    inicio_linha = time.time()

    try:
        df_result = buscar_endereco_com_fallback(
            endereco=logradouro,
            numero=numero,
            lat_origem=lat_origem,
            lon_origem=lon_origem,
            cadastro_path=cadastro_path,
            min_sim=min_sim,
            top_k=top_k
        )

        for col in ["similaridade", "linha_origem", "distancia_km"]:
            if col not in df_result.columns:
                df_result[col] = None

        df_result["linha_origem"] = i + 1

        # Anexa colunas originais
        for col in row.index:
            if col not in df_result.columns:
                df_result[col] = row[col]

        resultados.append(df_result)

    except Exception as e:
        print(f"   ❌ Erro ao processar '{logradouro}': {e}")

    print(f"   ⏱ Tempo: {round(time.time() - inicio_linha, 2)}s\n")

print(f"⏱ Tempo total: {round((time.time() - inicio_total)/60, 2)} min")

# -------------------------------------------
# 🔹 Consolidação e salvamento
# -------------------------------------------
if resultados:
    df_final = pd.concat(resultados, ignore_index=True)
    print(f"\n✅ {len(df_final)} registros processados com sucesso.")
else:
    df_final = pd.DataFrame()
    print("\n⚠️ Nenhum resultado obtido.")

if salvar_resultado and not df_final.empty:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    caminho_saida = os.path.splitext(caminho_entrada)[0] + f"_resultado_{timestamp}.xlsx"
    df_final.to_excel(caminho_saida, index=False)
    print(f"\n💾 Resultado salvo em:\n{caminho_saida}")



📂 Lendo planilha: C:\Users\vitor.braz\Downloads\sinistros_sp_parte1.xlsx

🌐 Definindo circunscrição via shapefile...


2025-10-30 10:44:54,237 - INFO - 🔍 Buscando no GEOCODE: RUA VIEIRA DE ALMEIDA%


✅ Circunscrição adicionada com sucesso.

🔍 Processando 5000 registros...

[1/5000] RUA VIEIRA DE ALMEIDA (-23.5944535766, -46.6141408221)


2025-10-30 10:44:58,002 - INFO - 🔍 Buscando no GEOCODE: RUA CURUCA%


   ⏱ Tempo: 3.77s

[2/5000] RUA CURUCA (-23.51325216, -46.58993175)


...

   ⏱ Tempo: 2.14s

[5/5000] AVENIDA INAJAR DE SOUZA (-23.495149, -46.684997)


2025-10-30 10:45:12,333 - INFO - 🔍 Buscando no GEOCODE: AVENIDA NOVE DE JULHO%


   ⏱ Tempo: 4.76s

[6/5000] AVENIDA NOVE DE JULHO (-23.56414174, -46.65681755)


2025-10-30 10:45:15,584 - INFO - 🔍 Buscando no GEOCODE: ESTRADA DOM JOAO NERY%


   ⏱ Tempo: 3.25s

[7/5000] ESTRADA DOM JOAO NERY (-23.5254571, -46.39850015)
